# D-MeZO — Bootstrap notebook for Google Colab Pro+

Target hardware: **RTX PRO 6000 Blackwell (96 GB)** или A100 80 GB.

Этот ноутбук:
1. Монтирует Google Drive (для чекпоинтов).
2. Клонирует или копирует проект `dmezo`.
3. Устанавливает зависимости.
4. Проверяет GPU.
5. Запускает Day 1 sanity check (MeZO на Qwen3-4B / SST-2).

**Бюджет:** ~30-50 compute units на полный прогон Day 1.

## 0. Mount Google Drive для чекпоинтов

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUNS_DIR = '/content/drive/MyDrive/dmezo_runs'
os.makedirs(RUNS_DIR, exist_ok=True)
print(f'Runs will be saved to: {RUNS_DIR}')

## 1. Клонировать проект из GitHub

Установи `GH_REPO` ниже на свой приватный repo (e.g. `your-username/dmezo`). Если repo приватный, добавь Personal Access Token через Colab Secrets (key icon в sidebar): создай secret `GH_TOKEN` с `repo` scope, в нём — твой PAT.

In [ ]:
GH_REPO = 'Siesher/dmezo'  # private repo on github.com

import os, shutil
if os.path.exists('/content/dmezo'):
    shutil.rmtree('/content/dmezo')

# Try to read PAT from Colab Secrets (for private repos).
try:
    from google.colab import userdata
    gh_token = userdata.get('GH_TOKEN')
    clone_url = f'https://{gh_token}@github.com/{GH_REPO}.git'
except Exception:
    clone_url = f'https://github.com/{GH_REPO}.git'  # works only if repo is public

!git clone {clone_url} /content/dmezo
%cd /content/dmezo
!git log --oneline -5

## 2. Установить зависимости

In [ ]:
!pip install -q --upgrade pip
!pip install -q -e .
# Flash attention (опционально, ускоряет MeZO forward на Blackwell):
!pip install -q flash-attn --no-build-isolation || echo 'flash-attn install skipped'

## 3. Проверить GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    total_gb = torch.cuda.mem_get_info()[1] / 1e9
    print(f'GPU: {name}')
    print(f'Memory: {free_gb:.1f} / {total_gb:.1f} GB free')
    print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')
    cc = torch.cuda.get_device_capability(0)
    print(f'Compute capability: {cc} (Blackwell = (10, 0) or higher)')

## 4. Запустить тесты

Должны пройти все: perturbation determinism + topology properties.

In [ ]:
!pytest tests/ -v

## 5. Day 1 sanity check — MeZO на Qwen3-4B / SST-2

Скрипт пишет JSONL-лог и чекпоинты в `experiments/01_sanity_qwen3_4b_sst2/` (внутри `/content/dmezo`). Чтобы сохранить в Drive, передадим `--output-dir`.

In [ ]:
OUTPUT_DIR = f'{RUNS_DIR}/01_sanity_qwen3_4b_sst2'
MLFLOW_URI = f'file:{RUNS_DIR}/mlruns'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_4b_sst2.yaml \
    --output-dir {OUTPUT_DIR} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-4b-sst2-day1

## 6. Анализ результата

## 6a. MLflow run summary

MLflow data persists in `{RUNS_DIR}/mlruns/` (на Drive). Можно посмотреть локально через `mlflow ui --backend-store-uri file:./mlruns` после скачивания, или прямо здесь в ноутбуке:

In [ ]:
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_sanity')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['attribute.start_time DESC'])
display(runs[['run_id', 'tags.mlflow.runName', 'tags.sanity_verdict',
              'metrics.init_eval_loss', 'metrics.final_eval_loss', 'metrics.eval_loss_drop_pct',
              'params.model.name', 'params.mezo.lr', 'params.mezo.eps']].head(10))

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

log_path = f'{OUTPUT_DIR}/log.jsonl'
rows = [json.loads(line) for line in open(log_path)]
df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train = df[df['train_loss'].notna()] if 'train_loss' in df.columns else df.iloc[0:0]
ev = df[df.get('eval_loss').notna()] if 'eval_loss' in df.columns else df.iloc[0:0]
if not train.empty:
    axes[0].plot(train['step'], train['train_loss'], label='train (moving avg)')
if not ev.empty:
    axes[0].plot(ev['step'], ev['eval_loss'], 'o-', label='eval')
axes[0].set_xlabel('MeZO step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Day 1 sanity: MeZO on Qwen3-4B / SST-2')
axes[0].legend()
axes[0].grid(alpha=0.3)

if 'projected_grad' in df.columns:
    pg = df[df['projected_grad'].notna()]
    axes[1].plot(pg['step'], pg['projected_grad'])
    axes[1].set_xlabel('MeZO step')
    axes[1].set_ylabel('Projected gradient')
    axes[1].set_title('Projected gradient (should be non-vanishing, non-NaN)')
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/sanity_plot.png', dpi=120)
plt.show()
print('Saved plot to', f'{OUTPUT_DIR}/sanity_plot.png')

## 7. Что дальше

Если sanity прошёл (eval loss упал на >10%):
- Переходить к Day 2 чтению FedKSeed/FedZeN.
- Запустить centralized baselines на BoolQ/COPA фоном.

Если sanity провалился:
- Проверить hyperparameters: `lr` в диапазоне 1e-7 - 1e-5, `eps=1e-3`.
- Проверить, что `param.requires_grad=True` для всех параметров.
- Распечатать `model.config` — убедиться, что Qwen3-4B загрузился именно как ожидалось.
- Запустить debug-версию с 10 steps и руками посмотреть на projected_grad.

## 8. Bonus: Qwen3.5-4B-Base — hybrid linear-attention sanity

Qwen3.5-4B-Base — это **vision-language модель с hybrid linear-attention text decoder** (32 слоя: 24 linear + 8 full attention, 3:1 ratio). Loader заморозит vision tower; MeZO будет perturbать только text decoder (~4.4B trainable params).

**Зачем:** Princeton MeZO 2023 проверил только на full-attention моделях. Это первый известный нам test MeZO на hybrid linear-attention арх. Любой результат публикабелен.

Если уже клонировал proj и хочешь подтянуть свежий код — сделай `!cd /content/dmezo && git pull` перед запуском ячейки ниже.

In [ ]:
OUTPUT_DIR_Q35 = f'{RUNS_DIR}/01_sanity_qwen3_5_4b_base_sst2'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_5_4b_base_sst2.yaml \
    --output-dir {OUTPUT_DIR_Q35} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-5-4b-base-sst2-hybrid-arch

In [ ]:
# Side-by-side: Qwen3-4B (Day 1) vs Qwen3.5-4B-Base (hybrid)
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_sanity')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['attribute.start_time DESC'])
cols = ['tags.mlflow.runName', 'params.model.name', 'tags.sanity_verdict',
        'metrics.init_eval_loss', 'metrics.final_eval_loss', 'metrics.eval_loss_drop_pct']
display(runs[cols].head(10))

## 9. Cross-dataset check — BoolQ on both architectures

После SST-2 Day 1 (Qwen3-4B PASS 88%) и Section 8 (Qwen3.5-4B-Base hybrid PASS 94.7%) — закрепляем finding на **BoolQ** (yes/no QA на длинных passages, SuperGLUE). Если оба тоже PASS — у нас 2×2 design (architecture × task), который перекрывает основной paper-claim risk "это специфика SST-2".

Сначала `!cd /content/dmezo && git pull` если notebook уже клонирован, иначе перезапусти runtime и re-run cells 1-6.

In [ ]:
!cd /content/dmezo && git pull

# Qwen3-4B / BoolQ (full attention baseline)
OUTPUT_DIR_Q3_BQ = f'{RUNS_DIR}/02_sanity_qwen3_4b_boolq'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_4b_boolq.yaml \
    --output-dir {OUTPUT_DIR_Q3_BQ} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-4b-boolq

In [ ]:
# Qwen3.5-4B-Base / BoolQ (hybrid linear-attention)
OUTPUT_DIR_Q35_BQ = f'{RUNS_DIR}/02_sanity_qwen3_5_4b_base_boolq'
!python scripts/01_sanity_check_mezo.py \
    --config configs/qwen3_5_4b_base_boolq.yaml \
    --output-dir {OUTPUT_DIR_Q35_BQ} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-5-4b-base-boolq-hybrid

## 10. Day 4 — first federated D-MeZO (2 clients, complete graph, weight_avg)

После centralized PASS на Qwen3-4B (Day 1, drop 88.1%) и кросс-арх validation (Section 8-9) — главный путь paper'а: **decentralized federated MeZO**. 2 клиента, complete topology, IID SST-2, weight-averaging consensus.

**Success criterion** (Week 1 plan): federated final eval ≤ centralized Day 1 × 1.1 = 0.18.

Memory budget: 2 × Qwen3-4B bf16 = 16 GB weights + activations ~25 GB peak. Blackwell 96 GB handles. Сделай `git pull` перед ячейкой.

In [ ]:
!cd /content/dmezo && git pull

OUTPUT_DIR_DAY4 = f'{RUNS_DIR}/03_dmezo_2c_complete_qwen3_4b_sst2'
!python scripts/03_dmezo_federated.py \
    --config configs/dmezo_2c_complete_qwen3_4b_sst2.yaml \
    --output-dir {OUTPUT_DIR_DAY4} \
    --mlflow-uri {MLFLOW_URI} \
    --run-name qwen3-4b-sst2-2c-complete-weight-avg

In [ ]:
# Day 4 vs Day 1 comparison: federated vs centralized on same model+task
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
runs_all = []
for exp_name in ['dmezo_sanity', 'dmezo_federated']:
    exp = mlflow.get_experiment_by_name(exp_name)
    if exp is None:
        continue
    runs_all.append(mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['attribute.start_time DESC']))
import pandas as pd
runs = pd.concat(runs_all, ignore_index=True)
cols = ['tags.mlflow.runName', 'params.model.name', 'params.data.task',
        'tags.algo', 'tags.n_clients', 'tags.topology', 'tags.consensus_mode',
        'tags.sanity_verdict', 'metrics.init_eval_loss', 'metrics.final_eval_loss',
        'metrics.eval_loss_drop_pct']
display(runs[[c for c in cols if c in runs.columns]].head(15))

In [ ]:
# 2x2 design summary: architecture x task
import mlflow
mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_sanity')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['attribute.start_time DESC'])
cols = ['tags.mlflow.runName', 'params.model.name', 'params.data.task',
        'tags.sanity_verdict', 'metrics.init_eval_loss', 'metrics.final_eval_loss',
        'metrics.eval_loss_drop_pct']
display(runs[cols].head(10))

## 11. Day 5 — federated D-MeZO grid (4 clients × topology × partition)

Цель Day 5 — впервые показать, что **federated MeZO работает на hybrid
linear-attention LLM** (Qwen3.5-4B-Base) при 4 клиентах с разными топологиями
и non-IID распределениями данных.

**Дизайн 2×2 grid** (4 запуска):

| topology | partition | rationale |
|---|---|---|
| complete(4) | IID | baseline — почти centralized (rho=0) |
| complete(4) | Dirichlet(0.5) | partition stress — Hsu 2019 convention |
| ring(4) | IID | topology stress — Koloskova 2020 rho~0.5 |
| ring(4) | Dirichlet(0.5) | worst cell — slow mixing + label skew |

Все runs: 1000 раундов, lr=3e-7, weight_avg consensus.

**Compute estimate:** 4 × ~10-12 min = ~45-50 min total, ~7-10 units.


In [ ]:
!cd /content/dmezo && git pull

CONFIGS = [
    'configs/dmezo_4c_complete_iid_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_complete_dir05_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_ring_iid_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2.yaml',
]
MLFLOW_URI = f'file:{RUNS_DIR}/mlruns'

for cfg_path in CONFIGS:
    # OUTPUT_DIR derives from the config's `output_dir` field, but we override
    # to land each run under Drive so it survives session loss.
    short = cfg_path.split('/')[-1].replace('.yaml', '')
    output_dir = f'{RUNS_DIR}/{short}'
    print(f'
=== {short} ===')
    !python scripts/03_dmezo_federated.py         --config {cfg_path}         --output-dir {output_dir}         --mlflow-uri {MLFLOW_URI}         --run-name {short}


In [ ]:
# Day 5 summary table — load all 4 runs from MLflow and compare.
import mlflow
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_federated_day5')
if exp is None:
    print('No experiment found — did the runs complete?')
else:
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    cols = [
        'tags.mlflow.runName',
        'params.federated.topology',
        'params.data.partition_mode',
        'metrics.init_eval_loss',
        'metrics.final_eval_loss',
        'metrics.eval_loss_drop_pct',
        'tags.sanity_verdict',
    ]
    cols_present = [c for c in cols if c in runs.columns]
    summary = runs[cols_present].sort_values('tags.mlflow.runName').reset_index(drop=True)
    # Compare against Day 4 centralized-equivalent (final 0.1793 on Qwen3-4B 2c).
    # Success criterion for Day 5: hybrid 4c <= 1.5 * Day 4 baseline = 0.27.
    DAY4_BASELINE = 0.1793
    DAY5_THRESHOLD = DAY4_BASELINE * 1.5
    print(f'Day 4 baseline (Qwen3-4B 2c complete IID): {DAY4_BASELINE:.4f}')
    print(f'Day 5 pass threshold (1.5x baseline):      {DAY5_THRESHOLD:.4f}')
    print()
    print(summary.to_string(index=False))


## 12. Day 6 — Nesterov-MeZO ablation (worst Day 5 cell)

Тестируем "N" в D-MeZO-N. Запускаем 2 варианта Nesterov-momentum на самом
сложном cell Day 5 (ring(4) + Dirichlet(α=0.5)):

- **Heavy-ball β=0.9** — стандартный momentum coefficient
- **Heavy-ball β=0.5** — пол-step momentum (safer для noisy ZO-grads)

Контроль = **Day 5 cell 4** (seed=42, без Nesterov, final=0.1381). Тот же seed
во всех Day 6 runs гарантирует bit-exact identical partition / MeZO seeds /
init — отличается ТОЛЬКО Nesterov ветвь.

**Гипотеза** (из `docs/03-algorithm-spec.md`): на heterogeneous partition со
slow consensus mixing Nesterov уменьшает client drift variance и даёт ~5-15%
улучшение. Falsifiable: если final ≥ 0.1381 — H0 не отвергаем.

Compute: 2 × ~37 мин ≈ 1.25 ч, ~5-7 units.


In [ ]:
!cd /content/dmezo && git pull

NEST_CONFIGS = [
    'configs/dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2_nest09.yaml',
    'configs/dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2_nest05.yaml',
]
MLFLOW_URI = f'file:{RUNS_DIR}/mlruns'

for cfg_path in NEST_CONFIGS:
    short = cfg_path.split('/')[-1].replace('.yaml', '')
    output_dir = f'{RUNS_DIR}/{short}'
    print(f'
=== {short} ===')
    !python scripts/03_dmezo_federated.py         --config {cfg_path}         --output-dir {output_dir}         --mlflow-uri {MLFLOW_URI}         --run-name {short}


In [ ]:
# Day 6 Nesterov ablation — compare against Day 5 cell 4 control.
import mlflow
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_URI)
CONTROL_FINAL = 0.1381   # Day 5 cell 4 (ring + Dir(0.5), nesterov off, seed=42)

day6 = mlflow.get_experiment_by_name('dmezo_federated_day6')
if day6 is None:
    print('No Day 6 experiment found yet.')
else:
    runs = mlflow.search_runs(experiment_ids=[day6.experiment_id])
    cols = [
        'tags.mlflow.runName',
        'params.nesterov.beta',
        'metrics.final_eval_loss',
        'metrics.eval_loss_drop_pct',
    ]
    cols_present = [c for c in cols if c in runs.columns]
    df = runs[cols_present].sort_values('tags.mlflow.runName').reset_index(drop=True)

    print(f'Control (Day 5 cell 4, nesterov OFF): {CONTROL_FINAL:.4f}')
    print()
    print(df.to_string(index=False))
    print()
    # Compute relative improvement per row.
    if 'metrics.final_eval_loss' in df.columns:
        for _, row in df.iterrows():
            final = row['metrics.final_eval_loss']
            improvement_pct = (CONTROL_FINAL - final) / CONTROL_FINAL * 100
            verdict = 'HELPS' if improvement_pct > 2 else ('NEUTRAL' if improvement_pct > -2 else 'HURTS')
            beta = row.get('params.nesterov.beta', '?')
            print(f"  beta={beta}: final={final:.4f}  delta={improvement_pct:+.2f}%  -> {verdict}")


## 13. Day 7 — empirical rigor pass

Закрываем gaps в эмпирике для one-pager:

**E (accuracy + clean centralized baseline)**
- Добавлена метрика `eval_acc` в скрипты — argmin-по-suffix-loss классификация. Логируется в MLflow на каждом eval-шаге.
- Новый `centralized_qwen3_5_4b_base_sst2_2000.yaml` — apples-to-apples reference точка для Day 5 federated runs.

**A (multi-seed)**
- Day 5 grid (4 configs) перезапускается на seed=42 (retrofit accuracy) и seed=43 (variance estimate). Среднее по двум seeds + диапазон даст error bars.

**Total compute:** 9 runs × ~37 min = ~5.5 h, ~15-20 units.

Запускай ячейки по очереди — между ними можно прерваться и продолжить позже,
все output_dir под Drive.


In [ ]:
# Phase 3a: centralized Qwen3.5 baseline (1 run)
!cd /content/dmezo && git pull

OUTPUT_DIR_C = f'{RUNS_DIR}/centralized_qwen3_5_4b_base_sst2_2000'
!python scripts/01_sanity_check_mezo.py     --config configs/centralized_qwen3_5_4b_base_sst2_2000.yaml     --output-dir {OUTPUT_DIR_C}     --mlflow-uri {MLFLOW_URI}     --run-name centralized_qwen3_5_4b_base_sst2_2000


In [ ]:
# Phase 3b: Day 5 grid retrofit at seed=42 (now with accuracy)
DAY5_CONFIGS = [
    'configs/dmezo_4c_complete_iid_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_complete_dir05_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_ring_iid_qwen3_5_4b_base_sst2.yaml',
    'configs/dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2.yaml',
]

for cfg_path in DAY5_CONFIGS:
    short = cfg_path.split('/')[-1].replace('.yaml', '') + '_s42'
    output_dir = f'{RUNS_DIR}/{short}'
    print(f'
=== {short} ===')
    !python scripts/03_dmezo_federated.py         --config {cfg_path}         --output-dir {output_dir}         --mlflow-uri {MLFLOW_URI}         --run-name {short}         --seed 42


In [ ]:
# Phase 3c: Day 5 grid at seed=43 (variance estimate)
for cfg_path in DAY5_CONFIGS:
    short = cfg_path.split('/')[-1].replace('.yaml', '') + '_s43'
    output_dir = f'{RUNS_DIR}/{short}'
    print(f'
=== {short} ===')
    !python scripts/03_dmezo_federated.py         --config {cfg_path}         --output-dir {output_dir}         --mlflow-uri {MLFLOW_URI}         --run-name {short}         --seed 43


In [ ]:
# Phase 3d: aggregate table — Day 5 grid mean + range across {42, 43}
import mlflow
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_URI)
exp = mlflow.get_experiment_by_name('dmezo_federated_day5')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])

# Extract grid runs (we tag by run_name suffix _s42 / _s43)
mask_grid = runs['tags.mlflow.runName'].str.endswith(('_s42', '_s43'), na=False)
grid = runs[mask_grid].copy()

# Strip the seed suffix to group configs.
grid['config'] = grid['tags.mlflow.runName'].str.replace(r'_s\d+$', '', regex=True)
grid['seed'] = grid['tags.mlflow.runName'].str.extract(r'_s(\d+)$').astype(int)

agg = grid.groupby('config').agg(
    final_loss_mean=('metrics.final_eval_loss', 'mean'),
    final_loss_std=('metrics.final_eval_loss', 'std'),
    final_loss_min=('metrics.final_eval_loss', 'min'),
    final_loss_max=('metrics.final_eval_loss', 'max'),
    final_acc_mean=('metrics.final_eval_acc', 'mean'),
    final_acc_std=('metrics.final_eval_acc', 'std'),
    n_seeds=('seed', 'count'),
).reset_index().sort_values('config')

print('Day 5 grid — aggregate over seeds {42, 43}:')
print(agg.to_string(index=False, float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))

# Also show the centralized baseline for comparison.
exp_c = mlflow.get_experiment_by_name('dmezo_centralized_baseline_day7')
if exp_c is not None:
    runs_c = mlflow.search_runs(experiment_ids=[exp_c.experiment_id])
    if len(runs_c) > 0:
        last = runs_c.iloc[0]
        print()
        print(f"Centralized baseline (Qwen3.5 / SST-2 / 2000 examples / seed=42):")
        print(f"  final_eval_loss = {last['metrics.final_eval_loss']:.4f}")
        print(f"  final_eval_acc  = {last.get('metrics.final_eval_acc', 'n/a')}")


## 14. HellaSwag — D-MeZO-N на сложной задаче (Qwen3.5-4B-Base)

Тестируем подход на **4-way commonsense reasoning** (Zellers 2019) — это
существенно сложнее SST-2/BoolQ, потому что концовки многотокенные и требуют
world knowledge, а не лексических сигналов.

**План (3 запуска):**

1. **Centralized baseline** — `qwen3_5_4b_base_hellaswag.yaml`. Vanilla MeZO, 1000 шагов.
   Цель: получить reference число для HellaSwag (paper Malladi 2023 не давал).
2. **D-MeZO-N v1** — `dmezo_4c_complete_iid_qwen3_5_4b_base_hellaswag_nest_decay.yaml`.
   4 клиента, complete graph, IID, β-decay 0.9→0 + clip=50. Это R1d-рецепт
   applied на HellaSwag — закрывает Theorem 3 эмпирически.
3. **Comparison** — federated vs centralized vs random (0.25). Success criterion:
   federated final_acc ≥ centralized final_acc × 0.9 (partition tax ≤ 10%).

**Compute estimate:** ~12-15 мин на каждый run × 2 = ~30 мин total, ~5 units.

Перед запуском: `!cd /content/dmezo && git pull` чтобы подтянуть HellaSwag
infrastructure (commit `80ba96f`).

In [ ]:
# 14a. Centralized HellaSwag baseline на Qwen3.5-4B-Base.
!cd /content/dmezo && git pull

OUTPUT_DIR_HS_C = f'{RUNS_DIR}/02_sanity_qwen3_5_4b_base_hellaswag'
!python scripts/01_sanity_check_mezo.py     --config configs/qwen3_5_4b_base_hellaswag.yaml     --output-dir {OUTPUT_DIR_HS_C}     --mlflow-uri {MLFLOW_URI}     --run-name qwen3-5-4b-base-hellaswag-centralized


In [ ]:
# 14b. D-MeZO-N v1 (β-decay 0.9→0 + clip50) на HellaSwag, 4 клиента.
!cd /content/dmezo && git pull

OUTPUT_DIR_HS_F = f'{RUNS_DIR}/08_dmezo_4c_complete_iid_qwen3_5_4b_base_hellaswag_nest_decay'
!python scripts/03_dmezo_federated.py     --config configs/dmezo_4c_complete_iid_qwen3_5_4b_base_hellaswag_nest_decay.yaml     --output-dir {OUTPUT_DIR_HS_F}     --mlflow-uri {MLFLOW_URI}     --run-name qwen3-5-4b-base-hellaswag-dmezo-n-v1


In [ ]:
# 14c. HellaSwag comparison — verdict + paper-ready table.
import mlflow
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_URI)

RANDOM_BASELINE = 0.25  # 4-way uniform random
PARTITION_TAX_LIMIT = 0.10  # federated should be within 10% of centralized

# Collect HellaSwag runs across experiments.
all_runs = []
for exp_name in ['dmezo_sanity', 'dmezo_federated_day8_rescue', 'dmezo_hellaswag_nest_decay',
                 'dmezo_hellaswag_baseline']:
    exp = mlflow.get_experiment_by_name(exp_name)
    if exp is None:
        continue
    r = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    if len(r) > 0:
        all_runs.append(r)

if not all_runs:
    print('No HellaSwag runs found — did 14a/14b complete?')
else:
    runs = pd.concat(all_runs, ignore_index=True)
    hs = runs[runs['tags.mlflow.runName'].str.contains('hellaswag', case=False, na=False)]

    cols = ['tags.mlflow.runName', 'params.data.task',
            'tags.algo', 'tags.n_clients', 'tags.topology',
            'metrics.init_eval_loss', 'metrics.final_eval_loss',
            'metrics.init_eval_acc', 'metrics.final_eval_acc',
            'metrics.eval_loss_drop_pct', 'tags.sanity_verdict']
    cols_present = [c for c in cols if c in hs.columns]
    summary = hs[cols_present].sort_values('tags.mlflow.runName').reset_index(drop=True)
    print('HellaSwag runs:')
    print(summary.to_string(index=False))
    print()
    print(f'Random-guess baseline (4-way uniform): {RANDOM_BASELINE:.4f}')

    # Compute verdict for federated vs centralized.
    centralized = hs[hs['tags.mlflow.runName'].str.contains('centralized', na=False)]
    federated = hs[hs['tags.mlflow.runName'].str.contains('dmezo-n', na=False)]
    if len(centralized) > 0 and len(federated) > 0:
        c_acc = centralized.iloc[0]['metrics.final_eval_acc']
        f_acc = federated.iloc[0]['metrics.final_eval_acc']
        tax = (c_acc - f_acc) / c_acc if c_acc > 0 else float('inf')
        print()
        print(f'Centralized final acc:  {c_acc:.4f}  (+{(c_acc-RANDOM_BASELINE)*100:.1f}pp vs random)')
        print(f'Federated  final acc:  {f_acc:.4f}  (+{(f_acc-RANDOM_BASELINE)*100:.1f}pp vs random)')
        print(f'Partition tax: {tax*100:+.1f}%  (limit: {PARTITION_TAX_LIMIT*100:.0f}%)')
        verdict = 'PASS' if (tax <= PARTITION_TAX_LIMIT and f_acc > RANDOM_BASELINE + 0.05) else 'FAIL'
        print(f'D-MeZO-N on HellaSwag verdict: {verdict}')


## 15. MathLogicQA — Russian symbolic reasoning (cross-lingual validation)

Финальная cross-domain валидация D-MeZO-N. После §14 (HellaSwag rescue) есть
очевидный следующий вопрос: **работает ли D-MeZO-N на reasoning на другом языке?**

**MathLogicQA** (часть MERA, `ai-forever/MERA`):
- 4-way symbolic logic + arithmetic reasoning **на русском**
- ~700 train / ~150 val examples
- Single-letter scoring (А/Б/В/Г) — MMLU/MERA стандарт
- Random-guess baseline = 25%

**План (2 запуска):**

1. **15a** — `qwen3_5_4b_base_mathlogicqa.yaml`: centralized vanilla MeZO baseline.
2. **15b** — `dmezo_4c_complete_iid_qwen3_5_4b_base_mathlogicqa_nest_decay.yaml`:
   D-MeZO-N v1 (β-decay + clip50), 4 clients complete IID.
3. **15c** — comparison + verdict (vanilla diverges? D-MeZO-N rescues?).

**Зачем:** покрываем 4 ортогональных domain (sentiment/QA/commonsense/symbolic
reasoning) × 2 языка (en/ru). Paper-claim максимально сильный.

Compute: ~30 min total. Перед запуском `!cd /content/dmezo && git pull`.

In [ ]:
# 15a. Centralized MathLogicQA baseline на Qwen3.5-4B-Base (vanilla MeZO).
!cd /content/dmezo && git pull

OUTPUT_DIR_ML_C = f'{RUNS_DIR}/03_sanity_qwen3_5_4b_base_mathlogicqa'
!python scripts/01_sanity_check_mezo.py     --config configs/qwen3_5_4b_base_mathlogicqa.yaml     --output-dir {OUTPUT_DIR_ML_C}     --mlflow-uri {MLFLOW_URI}     --run-name qwen3-5-4b-base-mathlogicqa-centralized


In [ ]:
# 15b. D-MeZO-N v1 (β-decay 0.9→0 + clip50) на MathLogicQA, 4 клиента.
!cd /content/dmezo && git pull

OUTPUT_DIR_ML_F = f'{RUNS_DIR}/09_dmezo_4c_complete_iid_qwen3_5_4b_base_mathlogicqa_nest_decay'
!python scripts/03_dmezo_federated.py     --config configs/dmezo_4c_complete_iid_qwen3_5_4b_base_mathlogicqa_nest_decay.yaml     --output-dir {OUTPUT_DIR_ML_F}     --mlflow-uri {MLFLOW_URI}     --run-name qwen3-5-4b-base-mathlogicqa-dmezo-n-v1


In [ ]:
# 15c. MathLogicQA comparison + verdict.
import mlflow
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_URI)
RANDOM_BASELINE = 0.25  # 4-way uniform random
PARTITION_TAX_LIMIT = 0.10

all_runs = []
for exp_name in ['dmezo_mathlogicqa_baseline', 'dmezo_mathlogicqa_nest_decay']:
    exp = mlflow.get_experiment_by_name(exp_name)
    if exp is None:
        continue
    r = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    if len(r) > 0:
        all_runs.append(r)

if not all_runs:
    print('No MathLogicQA runs found — did 15a/15b complete?')
else:
    runs = pd.concat(all_runs, ignore_index=True)
    ml = runs[runs['tags.mlflow.runName'].str.contains('mathlogicqa', case=False, na=False)]
    cols = ['tags.mlflow.runName',
            'metrics.init_eval_loss', 'metrics.final_eval_loss',
            'metrics.init_eval_acc', 'metrics.final_eval_acc',
            'metrics.eval_loss_drop_pct', 'tags.sanity_verdict']
    cols_present = [c for c in cols if c in ml.columns]
    summary = ml[cols_present].sort_values('tags.mlflow.runName').reset_index(drop=True)
    print('MathLogicQA runs:')
    print(summary.to_string(index=False))
    print()
    print(f'Random-guess baseline (4-way uniform): {RANDOM_BASELINE:.4f}')

    centralized = ml[ml['tags.mlflow.runName'].str.contains('centralized', na=False)]
    federated = ml[ml['tags.mlflow.runName'].str.contains('dmezo-n', na=False)]
    if len(centralized) > 0 and len(federated) > 0:
        c_acc = centralized.iloc[0]['metrics.final_eval_acc']
        f_acc = federated.iloc[0]['metrics.final_eval_acc']
        c_loss = centralized.iloc[0]['metrics.final_eval_loss']
        f_loss = federated.iloc[0]['metrics.final_eval_loss']
        tax = (c_acc - f_acc) / c_acc if c_acc > 0 else float('inf')
        print()
        print(f'Centralized: final loss={c_loss:.4f}  acc={c_acc:.4f}  (+{(c_acc-RANDOM_BASELINE)*100:.1f}pp vs random)')
        print(f'Federated:  final loss={f_loss:.4f}  acc={f_acc:.4f}  (+{(f_acc-RANDOM_BASELINE)*100:.1f}pp vs random)')
        print(f'Δ federated vs centralized: loss {(f_loss-c_loss)/c_loss*100:+.1f}%  acc {(f_acc-c_acc)*100:+.2f}pp')
        # Rescue verdict (matching HellaSwag pattern):
        c_init_acc = centralized.iloc[0].get('metrics.init_eval_acc', None)
        if c_init_acc is not None:
            c_divergent = c_acc < c_init_acc - 0.005  # 0.5pp threshold
            f_improved = f_acc > c_init_acc + 0.005
            if c_divergent and f_improved:
                print(f'>>> RESCUE PATTERN: vanilla DIVERGED, D-MeZO-N IMPROVED — matches HellaSwag finding')
            elif f_acc > c_acc:
                print(f'>>> Federated > Centralized — D-MeZO-N works on Russian symbolic reasoning')
            else:
                print(f'>>> No clear rescue pattern — verify hyperparameters or task difficulty')


## 16. MD-D-MeZO-N ablation — K=3 multi-direction SPSA (Gemini-критика)

Закрываем критику Gemini о том, что ρ-clipping — «костыль», и что
multi-direction SPSA (K независимых направлений на шаг) даст ускорение без bias.

**План (2 запуска × ~110 мин):**

1. **16a** — `dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2_nest_decay_k3.yaml`
   = Day 8 R1d (worst Day 5 cell) + K=3.
   Контроль: R1d final 0.1291. **PASS если K=3 < 0.1291**; NEUTRAL если ±5%.
2. **16b** — `dmezo_4c_complete_iid_qwen3_4b_hellaswag_nest_decay_k3.yaml`
   = HellaSwag federated + K=3.
   Контроль: 2026-05-18 federated final acc 0.7000. **PASS если K=3 acc > 0.70**.
3. **16c** — Auto-verdict comparing R1d vs K=3 + HellaSwag vs K=3.

Compute: K=3 даёт 2K=6 forwards/round → ~3× wall-clock vs baseline.
Каждый run ~50-60 мин на Blackwell. Total ~2 ч, ~15 units.

Перед запуском: `git pull` чтобы получить MD-D-MeZO-N infra (commit с
`md_mezo_step`/`md_nesterov_step`).

In [ ]:
# 16a. K=3 ablation на Day 5 worst cell — closes Gemini critique on SST-2.
!cd /content/dmezo && git pull

OUTPUT_DIR_K3_SST2 = f'{RUNS_DIR}/10_dmezo_md_k3_ring_dir05_qwen3_5_4b_base_sst2'
!python scripts/03_dmezo_federated.py     --config configs/dmezo_4c_ring_dir05_qwen3_5_4b_base_sst2_nest_decay_k3.yaml     --output-dir {OUTPUT_DIR_K3_SST2}     --mlflow-uri {MLFLOW_URI}     --run-name md-k3-sst2-ring-dir05-r1d-ablation


In [ ]:
# 16b. K=3 ablation на HellaSwag federated (Qwen3-4B).
!cd /content/dmezo && git pull

OUTPUT_DIR_K3_HS = f'{RUNS_DIR}/11_dmezo_md_k3_complete_iid_qwen3_4b_hellaswag'
!python scripts/03_dmezo_federated.py     --config configs/dmezo_4c_complete_iid_qwen3_4b_hellaswag_nest_decay_k3.yaml     --output-dir {OUTPUT_DIR_K3_HS}     --mlflow-uri {MLFLOW_URI}     --run-name md-k3-hellaswag-complete-iid-ablation


In [ ]:
# 16c. MD-D-MeZO-N verdict: K=3 vs K=1 controls.
import mlflow
import pandas as pd

mlflow.set_tracking_uri(MLFLOW_URI)

# Reference numbers from prior runs (from MEMORY / paper §5):
R1D_FINAL = 0.1291         # Day 8 R1d
HELLASWAG_FED_ACC = 0.7000  # 2026-05-18 federated D-MeZO-N v1 final acc

all_runs = []
for exp in ['dmezo_md_ablation', 'dmezo_federated_day8_rescue', 'dmezo_hellaswag_nest_decay']:
    e = mlflow.get_experiment_by_name(exp)
    if e is None:
        continue
    r = mlflow.search_runs(experiment_ids=[e.experiment_id])
    if len(r) > 0:
        all_runs.append(r)

if not all_runs:
    print('No MD-ablation runs found — did 16a/16b complete?')
else:
    runs = pd.concat(all_runs, ignore_index=True)
    md = runs[runs['tags.mlflow.runName'].str.contains('md-k3', case=False, na=False)]
    cols = ['tags.mlflow.runName', 'params.mezo.k_directions',
            'metrics.final_eval_loss', 'metrics.final_eval_acc',
            'metrics.eval_loss_drop_pct', 'tags.sanity_verdict']
    cols_present = [c for c in cols if c in md.columns]
    print('K=3 MD-D-MeZO-N runs:')
    print(md[cols_present].to_string(index=False))
    print()

    # SST-2 K=3 vs R1d.
    k3_sst2 = md[md['tags.mlflow.runName'].str.contains('sst2', case=False, na=False)]
    if len(k3_sst2) > 0:
        k3_final = k3_sst2.iloc[0]['metrics.final_eval_loss']
        delta = (k3_final - R1D_FINAL) / R1D_FINAL * 100
        verdict = ('PASS (variance reduction helps)' if delta < -2 else
                   ('NEUTRAL (ρ-clip alone sufficient)' if abs(delta) < 5 else
                    'FAIL (multi-direction hurts)'))
        print(f'SST-2 K=3 vs R1d: {k3_final:.4f} vs {R1D_FINAL:.4f} ({delta:+.2f}%) → {verdict}')

    # HellaSwag K=3 vs federated v1.
    k3_hs = md[md['tags.mlflow.runName'].str.contains('hellaswag', case=False, na=False)]
    if len(k3_hs) > 0:
        k3_acc = k3_hs.iloc[0]['metrics.final_eval_acc']
        delta_pp = (k3_acc - HELLASWAG_FED_ACC) * 100
        verdict = ('PASS (K=3 boosts rescue)' if delta_pp > 2 else
                   ('NEUTRAL (rescue saturated)' if abs(delta_pp) < 2 else
                    'FAIL (K=3 worsens rescue)'))
        print(f'HellaSwag K=3 vs federated v1: {k3_acc:.4f} vs {HELLASWAG_FED_ACC:.4f} ({delta_pp:+.2f}pp) → {verdict}')


## 17. ε warmup autotuner — cross-arch sweep

Запускаем warmup-based ε autotuner на 4 архитектурах для проверки
гипотезы «оптимальный ε зависит от модели». Локально (Qwen3-0.6B,
Qwen3.5-0.8B) уже прогнано — здесь Colab для middle/large моделей.

**Метрика:** $J(arepsilon) = 	ilde{m bias} + 	ilde{m var}$ (min-max
normalised). Bias proxy через 2nd-order Taylor $(L_+ + L_- - 2L_0)/arepsilon^2
pprox z^	op H z$. Variance proxy — std ρ̂ при свежем z на каждый probe.

**Default grid:** ε ∈ {1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2}, 30 probes/ε.

Compute: ~3 минуты на Qwen3-4B на Blackwell (8 candidates × 30 probes × 2 forwards = 480 forwards).

In [ ]:
# 17a. Qwen3-4B (full-attn) — middle-size baseline.
!cd /content/dmezo && git pull

!python scripts/diagnose_eps_warmup.py     --model Qwen/Qwen3-4B     --dtype bfloat16     --n-probes 30


In [ ]:
# 17b. Qwen3.5-4B-Base (hybrid linear-attn) — наша main paper arch.
!cd /content/dmezo && git pull

!python scripts/diagnose_eps_warmup.py     --model Qwen/Qwen3.5-4B-Base     --dtype bfloat16     --n-probes 30


In [ ]:
# 17c. Cross-arch comparison plot.
import json
import matplotlib.pyplot as plt
import numpy as np

models = [
    ('Qwen/Qwen3-0.6B',       'Qwen3-0.6B (full-attn)',      '#1f77b4'),
    ('Qwen/Qwen3.5-0.8B',     'Qwen3.5-0.8B (hybrid lin-attn)', '#ff7f0e'),
    ('Qwen/Qwen3-4B',         'Qwen3-4B (full-attn)',        '#2ca02c'),
    ('Qwen/Qwen3.5-4B-Base',  'Qwen3.5-4B-Base (hybrid)',    '#d62728'),
]
DIAG_DIR = '/content/dmezo/experiments/diagnostics'

fig, ax = plt.subplots(figsize=(11, 5))
for model_name, label, color in models:
    short = model_name.replace('/', '_').replace('.', 'p')
    path = f'{DIAG_DIR}/eps_warmup_{short}.json'
    try:
        d = json.load(open(path))
        eps_vals = sorted([s['eps'] for s in d['scores'].values()])
        scores = {s['eps']: s for s in d['scores'].values()}
        j_norm = [scores[e]['j_norm'] for e in eps_vals]
        ax.semilogx(eps_vals, j_norm, 'o-', color=color, label=f"{label}, ε*={d['eps_star']:.1e}")
    except FileNotFoundError:
        print(f'WARNING: {path} not found — did 17a/17b/local runs complete?')

ax.set_xlabel(r'$arepsilon$', fontsize=12)
ax.set_ylabel(r'$J(arepsilon)$ (bias + variance, normalised)', fontsize=12)
ax.set_title('ε* across architectures (lower J = better trade-off)')
ax.axvline(1e-3, color='gray', linestyle=':', label='Princeton default (1e-3)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dmezo_runs/fig9_cross_arch_eps.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved cross-arch comparison.')


## 18. lr × ε joint sweep on HellaSwag / Qwen3.5-4B-BaseЦель: оценить совместное влияние learning rate и ε(t)-schedule на reasoning task. Из памяти проекта мы знаем что vanilla MeZO ДИВЕРГИРУЕТ на HellaSwag (−2.5pp acc), а D-MeZO-N конвергирует (+3.75pp acc). Здесь — joint sweep по lr и ε для обоих вариантов, чтобы понять (a) есть ли «sweet spot» в (lr, ε) пространстве для vanilla, (b) рекомендации по ε-schedule переносятся ли с SST-2 на reasoning.**Grid**: lr ∈ {1e-7, 3e-7, 1e-6, 3e-6} × ε_schedule ∈ {const_1e-3, decay 1e-3→1e-4, warmup 1e-2→1e-3} × variant ∈ {vanilla, dmezo_n}. **500 шагов**, eval каждые 50 шагов на 100-example HellaSwag validation. По 12 cells на variant, всего 24. Ожидаемое время на Blackwell ≈ 60-90 мин.

### 18a. (опционально) Ускорение через flash-linear-attentionQwen3.5 hybrid arch использует кастомные линейные attention layers. Без `flash-linear-attention` + `causal-conv1d` (slow torch fallback) forward в 5-10× медленнее. Если упадёт установка — игнорируем, sweep всё равно отработает.

In [ ]:
# 18a. Try to install fla fast-path (optional speedup for hybrid arch)import subprocesstry:    r = subprocess.run(        ["pip", "install", "flash-linear-attention", "causal-conv1d", "--quiet"],        capture_output=True, text=True, timeout=300,    )    print(r.stdout[-500:] if r.stdout else "(no stdout)")    if r.returncode != 0:        print("install non-zero exit; falling back to slow torch impl")        print(r.stderr[-500:])    else:        print("fla installed OK — linear-attn fast path enabled")except Exception as e:    print(f"fla install failed (will use slow torch impl): {e}")

### 18b. Vanilla MeZO sweep (control, expected to diverge on HellaSwag)

In [ ]:
# 18b. Vanilla MeZO — 12 cells × 500 steps. Expected ~30-45 min on Blackwell.!cd /content/dmezo && git pull!cd /content/dmezo && python scripts/sweep_lr_eps_hellaswag.py     --model Qwen/Qwen3.5-4B-Base     --variant vanilla     --lrs 1e-7 3e-7 1e-6 3e-6     --schedules const_1e-3 exp_decay_1e-3_to_1e-4 exp_decay_1e-2_to_1e-3     --steps 500     --eval-every 50     --eval-batches 25     --dtype bfloat16

### 18c. D-MeZO-N sweep (heavy-ball β-decay 0.9→0 + ρ-clip C=50)

In [ ]:
# 18c. D-MeZO-N — same grid. Expected ~30-45 min on Blackwell.!cd /content/dmezo && python scripts/sweep_lr_eps_hellaswag.py     --model Qwen/Qwen3.5-4B-Base     --variant dmezo_n     --lrs 1e-7 3e-7 1e-6 3e-6     --schedules const_1e-3 exp_decay_1e-3_to_1e-4 exp_decay_1e-2_to_1e-3     --steps 500     --eval-every 50     --eval-batches 25     --dtype bfloat16     --rho-clip 50.0     --beta-start 0.9     --beta-end 0.0

### 18d. Persist sweep results to Drive

In [ ]:
# 18d. Copy JSON + figures to Drive for offline analysis.import shutilfrom pathlib import Pathdrive_dest = Path('/content/drive/MyDrive/dmezo_runs/sweep_lr_eps_hellaswag')drive_dest.mkdir(parents=True, exist_ok=True)artefacts = [    Path('/content/dmezo/experiments/diagnostics/sweep_lr_eps_hellaswag_vanilla_Qwen_Qwen3p5-4B-Base.json'),    Path('/content/dmezo/experiments/diagnostics/sweep_lr_eps_hellaswag_dmezo_n_Qwen_Qwen3p5-4B-Base.json'),    Path('/content/dmezo/docs/figures/fig16_sweep_lr_eps_hellaswag_vanilla_Qwen_Qwen3p5-4B-Base.png'),    Path('/content/dmezo/docs/figures/fig16_sweep_lr_eps_hellaswag_dmezo_n_Qwen_Qwen3p5-4B-Base.png'),]for src in artefacts:    if src.exists():        shutil.copy(src, drive_dest / src.name)        print(f"  copied {src.name}  ({src.stat().st_size / 1024:.1f} KB)")    else:        print(f"  MISSING: {src.name}")print(f"Done. Files in: {drive_dest}")